# TCN Experiments: Handcrafted Feature Results

This notebook runs the TCN experiments from our paper on predicting multi-state behavior in MD simulations.

It evaluates TCN models on several feature sets under **protein-level holdout** (within-family generalization) on the first 50% of the the trajectory:

| Feature Set | Dimensionality  | Description |
|-------------|----------------|-------------|
| Scalar | 3 | RMSD, radius of gyration, TM3-TM6 distance |
| TICA projections | 5  | Top-5 TICA slow mode projections |
| Combined | 58  | All handcrafted features concatenated (Graph, Structural, Scalar, TICA)|
| Autocorrelation | 52 | Per-frame sliding window of residue RMST, autocorrelation at lags 1 and 5 |
| Oracle (50%) | 7 | Features derived from labeling pipeline (upper bound) |

## Prerequisites

1. **Environment**: `conda activate cs229_md_project`
2. **GCS Mount**: Data must be accessible via gcsfuse (see setup below)
3. **Working Directory**: Run from `cs229_md_prediction/src/models/TCN/`

## Setup

In [1]:
import os
import sys
import json
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

# Verify environment
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Working directory: {os.getcwd()}")

Python: 3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
PyTorch: 2.10.0+cpu
CUDA available: False
Working directory: /home/persav/git/cs229-md-prediction/src/models/TCN


In [ ]:
# =============================================================================
# CONFIGURATION 
# =============================================================================

# Project root (adjust if running from different directory)
PROJECT_ROOT = Path("../../..").resolve()

# GCS mount point - where gcsfuse mounts the bucket
GCS_MOUNT = PROJECT_ROOT / "data" / "gcs_mount"

# Alternative: direct GCS path if running on Vertex AI Workbench
# GCS_MOUNT = Path("/home/jupyter/gcs_mount")

# Metadata and split files
METADATA_DIR = PROJECT_ROOT / "data" / "metadata" / "splits"

# Results output directory
RESULTS_DIR = PROJECT_ROOT / "results" / "tcn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths exist
print("Checking paths...")
print(f"  PROJECT_ROOT: {PROJECT_ROOT} {'Yes' if PROJECT_ROOT.exists() else 'No'}")
print(f"  GCS_MOUNT: {GCS_MOUNT} {'Yes' if GCS_MOUNT.exists() else 'No'}")
print(f"  METADATA_DIR: {METADATA_DIR} {'Yes' if METADATA_DIR.exists() else 'No'}")
print(f"  RESULTS_DIR: {RESULTS_DIR} {'Yes' if RESULTS_DIR.exists() else 'No'}")

Checking paths...
  PROJECT_ROOT: /home/persav/git/cs229-md-prediction Yes
  GCS_MOUNT: /home/persav/git/cs229-md-prediction/data/gcs_mount Yes
  METADATA_DIR: /home/persav/git/cs229-md-prediction/data/metadata/splits Yes
  RESULTS_DIR: /home/persav/git/cs229-md-prediction/results/tcn Yes


In [20]:
# Verify GCS mount has expected data
SCALAR_DIR = GCS_MOUNT / "data" / "processed" / "v4" / "features_50pct" / "scalar"
TICA_DIR = GCS_MOUNT / "data" / "processed" / "v4" / "features_50pct" / "tica" / "projections"
ORACLE_DIR = GCS_MOUNT / "data" / "processed" / "v4" / "features_50pct" / "sanity_check_features"
AUTOCORRELATION_DIR = GCS_MOUNT / "data" / "processed" / "autocorr"
COMBINED_DIR = GCS_MOUNT / "data" / "processed" / "combined_features" / "combined_50pct"

expected_dirs = [
    SCALAR_DIR,
    TICA_DIR,
    ORACLE_DIR,
    AUTOCORRELATION_DIR,
    COMBINED_DIR,
]

print("Checking feature directories...")
for d in expected_dirs:
    exists = d.exists()
    n_files = len(list(d.glob("*.npy"))) if exists else 0
    status = f"({n_files} files)" if exists else " MISSING"
    print(f"  {d.name}: {status}")

if not all(d.exists() for d in expected_dirs):
    print("\n  Some directories are missing. Ensure gcsfuse is mounted")

Checking feature directories...
  scalar: (246 files)
  projections: (243 files)
  sanity_check_features: (246 files)
  autocorr: (246 files)
  combined_50pct: (246 files)


In [5]:
train_csv = METADATA_DIR / "protein_level_train.csv"
test_csv = METADATA_DIR / "protein_level_test.csv"

## Experiment Configuration

Define the experiments to run. Each experiment specifies:
- Feature directory path
- Output label for results
- Number of seeds for variance estimation

In [ ]:
# =============================================================================
# EXPERIMENT DEFINITIONS
# =============================================================================

EXPERIMENTS = {
    # Handcrafted features (expect modest/degenerate performance)
    "scalar_50pct": {
        "npy_dir": SCALAR_DIR,
        "label": "Scalar",
    },
    "tica_50pct": {
        "npy_dir": TICA_DIR,
        "label": "TICA projections (5-dim)",
    },
    "combined_50pct": {
        "npy_dir": COMBINED_DIR,
        "label": "Combined (58-dim)",
    },
    
    "autocorr_50pct": {
        "npy_dir": AUTOCORRELATION_DIR,
        "label": "autocorrelation",
    },
}

# Training hyperparameters (fixed across experiments)
TCN_CONFIG = {
    "batch_size": 1,
    "num_workers": 0,
    "patience": 50,
    "epochs": 200,
    "lr": 0.001,
    "n_seeds": 1,  # Number of random seeds for variance estimation
}

print(f"Configured {len(EXPERIMENTS)} experiments with {TCN_CONFIG['n_seeds']} seeds each")

Configured 3 experiments with 1 seeds each


## Run TCN Training

The following cells train TCN models for each feature set. Each experiment runs multiple seeds for variance estimation.

**Estimated runtime**: ~5-10 minutes per experiment on CPU, ~1-2 minutes on GPU.

In [ ]:
def run_tcn_experiment(exp_name, exp_config, seeds=3, config=TCN_CONFIG, dry_run=False):
    """
    Run TCN training for a single experiment across multiple seeds.
    
    Args:
        exp_name: Experiment identifier
        exp_config: Dict with 'npy_dir' and 'label'
        seeds: Number of random seeds to run
        dry_run: If True, print command without executing
    
    Returns:
        List of result directories
    """
    output_base = RESULTS_DIR / exp_name
    output_base.mkdir(parents=True, exist_ok=True)
    
    result_dirs = []
    
    for seed in range(seeds):
        cmd = f"""
python train_tcn.py \
    --train_csv {train_csv} \
    --test_csv {test_csv} \
    --npy_dir {exp_config['npy_dir']} \
    --batch_size {config['batch_size']} \
    --num_workers {config['num_workers']} \
    --patience {config['patience']} \
    --epochs {config['epochs']} \
    --lr {config['lr']} \
    --output_dir {output_base}
""".strip()
        
        print(f"\n{'='*60}")
        print(f"Running: {exp_config['label']} (seed {seed})")
        print(f"{'='*60}")
        
        if dry_run:
            print(f"[DRY RUN] Would execute:\n{cmd}")
        else:
            os.system(cmd)
        
        # Find the created result directory
        result_dirs.extend(sorted(output_base.glob("tcn_*")))
    
    return result_dirs

In [13]:
# =============================================================================
# RUN ALL EXPERIMENTS
# =============================================================================
# Set dry_run=True to preview commands without executing

DRY_RUN = False  # Set to False to actually run experiments

all_result_dirs = {}

for exp_name, exp_config in EXPERIMENTS.items():
    # Check if feature directory exists
    if not exp_config['npy_dir'].exists():
        print(f"  Skipping {exp_name}: feature directory not found")
        continue
    
    result_dirs = run_tcn_experiment(
        exp_name, 
        exp_config, 
        seeds=TCN_CONFIG['n_seeds'],
        config=TCN_CONFIG,
        dry_run=DRY_RUN
    )
    all_result_dirs[exp_name] = result_dirs

print(f"\n\nCompleted {len(all_result_dirs)} experiments")


Running: Scalar (seed 0)
Using device: cpu
Output directory: /home/persav/git/cs229-md-prediction/results/tcn/scalar_50pct/tcn_20260313_203910

Loading datasets...
Loaded 204 samples
  Multi: 62.0 (30.4%)
  Single: 142 (69.6%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 3 features from data

Initializing TCN with 3 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        13,676
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=27.0954, Bal Acc=0.500, AUROC=0.494, AP=0.302
  Test:  Loss=20.9302, Bal Acc=0.500, AUROC=0.500, AP=0.209
  Skipped batches due to no valid samples: 1
   New best AUROC: 0.500
Epoch 2/200
  Train: Loss=30.3928, Bal Acc=0.500, AUROC=0.489, AP=0.305
  Test:  Loss=20.9302, Bal Acc=0.500, AUROC=0.500, AP=0.209
  Skipped batches due t

/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p


Running: TICA projections (5-dim) (seed 0)
Using device: cpu
Output directory: /home/persav/git/cs229-md-prediction/results/tcn/tica_50pct/tcn_20260313_211625

Loading datasets...
Loaded 204 samples
  Multi: 62.0 (30.4%)
  Single: 142 (69.6%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 5 features from data

Initializing TCN with 5 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        13,876
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=1.5851, Bal Acc=0.484, AUROC=0.468, AP=0.292
  Test:  Loss=0.8399, Bal Acc=0.500, AUROC=0.801, AP=0.657
  Skipped batches due to no valid samples: 4
   New best AUROC: 0.801
Epoch 2/200
  Train: Loss=1.1494, Bal Acc=0.500, AUROC=0.530, AP=0.345
  Test:  Loss=0.6409, Bal Acc=0.500, AUROC=0.748, AP=0.608
  Skipped b

/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p


Running: Combined (58-dim) (seed 0)
Using device: cpu
Output directory: /home/persav/git/cs229-md-prediction/results/tcn/combined_50pct/tcn_20260313_212711

Loading datasets...
Loaded 204 samples
  Multi: 62.0 (30.4%)
  Single: 142 (69.6%)
  Using frac=1.0 of each trajectory
Loaded 43 samples
  Multi: 9.0 (20.9%)
  Single: 34 (79.1%)
  Using frac=1.0 of each trajectory
  Auto-detected 58 features from data

Initializing TCN with 58 input features...
TCN Summary:
  Receptive field:   61 timesteps
  Pooling:           mean
  Parameters:        19,176
  Architecture:      [25, 25, 25, 25]

Using pos_weight=1.0 for imbalanced data

Starting training...
Epoch 1/200
  Train: Loss=30.3922, Bal Acc=0.500, AUROC=0.500, AP=0.305
  Test:  Loss=20.9302, Bal Acc=0.500, AUROC=0.500, AP=0.209
  Skipped batches due to no valid samples: 1
   New best AUROC: 0.500
Epoch 2/200
  Train: Loss=29.5444, Bal Acc=0.500, AUROC=0.514, AP=0.304
  Test:  Loss=14.8357, Bal Acc=0.500, AUROC=0.624, AP=0.340
  Skippe

/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/persav/miniconda3/envs/cs229_md_project/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_p



Completed 3 experiments


## Collect and Analyze Results

In [16]:
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score,
    average_precision_score, f1_score, confusion_matrix
)

def collect_results(base_dir, label):
    """
    Collect metrics across seed subdirectories.
    Returns dict with mean/std for each metric.
    """
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return None
    
    # Find all seed runs
    run_dirs = sorted(base_dir.glob("tcn_*"))
    if not run_dirs:
        return None
    
    metrics = {'auroc': [], 'bal_acc': [], 'ap': [], 'f1': []}
    
    for run_dir in run_dirs:
        # Try metrics.json first
        metrics_file = run_dir / "metrics.json"
        if metrics_file.exists():
            with open(metrics_file) as f:
                m = json.load(f)
            metrics['auroc'].append(m.get('test_auroc', m.get('auroc', np.nan)))
            metrics['bal_acc'].append(m.get('test_balanced_accuracy', m.get('balanced_accuracy', np.nan)))
            metrics['ap'].append(m.get('test_avg_precision', m.get('avg_precision', np.nan)))
            metrics['f1'].append(m.get('test_f1', m.get('f1', np.nan)))
        
        # Fallback to test_predictions.csv
        elif (run_dir / "test_predictions.csv").exists():
            df = pd.read_csv(run_dir / "test_predictions.csv")
            y_true = df['true_label'].values
            y_prob = df['pred_prob'].values
            y_pred = (y_prob > 0.5).astype(int)
            
            metrics['auroc'].append(roc_auc_score(y_true, y_prob))
            metrics['bal_acc'].append(balanced_accuracy_score(y_true, y_pred))
            metrics['ap'].append(average_precision_score(y_true, y_prob))
            metrics['f1'].append(f1_score(y_true, y_pred))
    
    if not metrics['auroc']:
        return None
    
    # Determine if predictions are "degenerate" (all same class)
    bal_acc_mean = np.nanmean(metrics['bal_acc'])
    behaviour = "Degenerate" if bal_acc_mean <= 0.52 else "Real signal"
    
    return {
        'label': label,
        'n_runs': len(metrics['auroc']),
        'auroc_mean': np.nanmean(metrics['auroc']),
        'auroc_std': np.nanstd(metrics['auroc']),
        'bal_acc_mean': bal_acc_mean,
        'bal_acc_std': np.nanstd(metrics['bal_acc']),
        'ap_mean': np.nanmean(metrics['ap']),
        'f1_mean': np.nanmean(metrics['f1']),
        'behaviour': behaviour,
    }

In [21]:
# =============================================================================
# COLLECT RESULTS FROM ALL EXPERIMENTS
# =============================================================================

all_results = []

for exp_name, exp_config in EXPERIMENTS.items():
    result_dir = RESULTS_DIR / exp_name
    result = collect_results(result_dir, exp_config['label'])
    
    if result:
        result['experiment'] = exp_name
        all_results.append(result)
        print(f"{exp_config['label']}: AUROC = {result['auroc_mean']:.3f} ± {result['auroc_std']:.3f} "
              f"({result['n_runs']} runs) [{result['behaviour']}]")
    else:
        print(f"{exp_config['label']}: No results found")

# Create summary DataFrame
if all_results:
    results_df = pd.DataFrame(all_results)
    print("\n" + "="*70)
    print(results_df[['label', 'auroc_mean', 'auroc_std', 'bal_acc_mean', 'behaviour']].to_string(index=False))

Scalar: AUROC = 0.863 ± 0.000 (1 runs) [Degenerate]
TICA projections (5-dim): AUROC = 0.801 ± 0.000 (1 runs) [Degenerate]
Combined (58-dim): AUROC = 0.761 ± 0.000 (1 runs) [Degenerate]

                   label  auroc_mean  auroc_std  bal_acc_mean  behaviour
                  Scalar    0.862745        0.0           0.5 Degenerate
TICA projections (5-dim)    0.800654        0.0           0.5 Degenerate
       Combined (58-dim)    0.761438        0.0           0.5 Degenerate


In [19]:
# Save results to JSON for reproducibility
if all_results:
    output_file = RESULTS_DIR / "tcn_reproduction_summary.json"
    with open(output_file, 'w') as f:
        json.dump({
            'timestamp': datetime.now().isoformat(),
            'config': TCN_CONFIG,
            'results': all_results,
        }, f, indent=2, default=str)
    print(f"Results saved to: {output_file}")

Results saved to: /home/persav/git/cs229-md-prediction/results/tcn/tcn_reproduction_summary.json
